## MLflow Exercise

This notebook reproduces the **MLflow monitoring example** from the text.  
It runs a simple OpenAI chat loop while **logging model parameters and metrics** (token counts, latency) to MLflow.

**Goals of this exercise**
- See how MLflow tracks metrics and parameters automatically.
- Practice instrumenting a small GenAI script.
- Observe results in the MLflow UI.

### Setup: imports, configuration, and client

- **Imports and initialization**: MLflow for experiment tracking, OpenAI for text generation, `tiktoken` for token counting, and `colorama` for colored terminal output.
- **API key loading**: Reads from an environment variable instead of hard-coding, which is a secure and portable practice.
- **Configuration constants**: Defines model name, temperature, token limits, etc.—this makes experiments reproducible.
- **MLflow setup**: Tries to connect to a local MLflow server (`localhost:5000`) and falls back to a local folder if unavailable.  
  This prevents the notebook from hanging if the MLflow service isn’t running.

In [15]:
import os, time, pathlib, requests, json, tempfile
import mlflow
from openai import OpenAI
import tiktoken as tk
from colorama import Fore, Style, init

# removes harmless "Filesystem tracking backend (e.g., './mlruns') is deprecated." warning 
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="mlflow")


# --- API key configuration ---
os.environ["OPENAI_API_BOOK_KEY"] = 'YOUR_KEY_HERE'
API_KEY = os.getenv("OPENAI_API_BOOK_KEY")
if not API_KEY:
    raise RuntimeError(
        "OPENAI_API_BOOK_KEY is not set.\n"
        "In PowerShell:\n"
        '  setx OPENAI_API_BOOK_KEY "sk-..."\n'
        "Then restart Jupyter, or temporarily set it above."
    )

# --- Model and logging settings ---
MODEL = "gpt-3.5-turbo"
MLFLOW_URI = "http://localhost:5000"
TEMPERATURE = 0.7
TOP_P = 1
FREQUENCY_PENALTY = 0
PRESENCE_PENALTY = 0
MAX_TOKENS = 800
DEBUG = False
EXPERIMENT_NAME = "GenAI_book"

# --- Initialize pretty output and client ---
init()
def print_user_input(text): print(f"{Fore.GREEN}You:{Style.RESET_ALL} {text}")
def print_ai_output(text):  print(f"{Fore.BLUE}AI Assistant:{Style.RESET_ALL} {text}")

client = OpenAI(api_key=API_KEY)

# --- Configure MLflow: connect to server or fall back to file store ---
def server_ok(url: str, timeout=1.0) -> bool:
    try:
        r = requests.get(f"{url}/api/2.0/mlflow/experiments/list", timeout=timeout)
        return r.status_code in (200, 401, 403)
    except Exception:
        return False

if server_ok(MLFLOW_URI):
    mlflow.set_tracking_uri(MLFLOW_URI)
else:
    local_dir = pathlib.Path.home() / "mlruns_book_fallback"
    local_dir.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(f"file:///{local_dir.as_posix()}")

mlflow.set_experiment(EXPERIMENT_NAME)
print("✅ MLflow tracking configured.")

✅ MLflow tracking configured.


### Generate Function

- **Makes the API call** to the OpenAI model.
- **Records latency** (how long the call takes).
- **Counts tokens** using `tiktoken` (with a safe fallback).
- **Logs metrics** like latency and token usage to MLflow.
- **Logs parameters** (model name, temperature, etc.) once per run.

In [8]:
def generate_text(conversation, max_tokens=100) -> str:
    """Send a chat to OpenAI, record latency, and log metrics."""
    # local safe token counter (so we never get NameError)
    def count_tokens_safe_local(text: str, encoding_name="cl100k_base"):
        try:
            import tiktoken
            enc = tiktoken.get_encoding(encoding_name)
            return len(enc.encode(text, disallowed_special=()))
        except Exception:
            return len(text.split())

    # --- call model and measure latency ---
    start_time = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=conversation,
        temperature=TEMPERATURE,
        max_tokens=max_tokens,
        top_p=TOP_P,
        frequency_penalty=FREQUENCY_PENALTY,
        presence_penalty=PRESENCE_PENALTY,
    )
    latency = time.time() - start_time
    message_response = response.choices[0].message.content

    # --- compute token counts ---
    last_content = conversation[-1]["content"] if conversation else ""
    prompt_tokens = count_tokens_safe_local(last_content)
    conversation_tokens = count_tokens_safe_local(str(conversation))
    completion_tokens = count_tokens_safe_local(message_response)

    # --- log metrics to MLflow ---
    mlflow.log_metrics({
        "request_latency": latency,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "conversation_tokens": conversation_tokens,
    })

    # --- log parameters (immutable for this run) ---
    mlflow.log_params({
        "model": MODEL,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "frequency_penalty": FREQUENCY_PENALTY,
        "presence_penalty": PRESENCE_PENALTY,
    })

    return message_response

### Simple Chat Loop

Defines a **lightweight chat loop** that collects messages, calls the model, and prints results in color.
- Runs two quick turns (“fun fact” + “shorter”) as a test conversation.
- Because we’re inside an MLflow run (`with mlflow.start_run()`), all the metrics and parameters from `generate_text()` are recorded automatically.

In [11]:
def chat_once(user_text: str, conversation=None, max_tokens=MAX_TOKENS):
    if conversation is None:
        conversation = [{"role": "system", "content": "You are a helpful assistant."}]
    print_user_input(user_text)
    conversation.append({"role": "user", "content": user_text})
    ai_output = generate_text(conversation, max_tokens)
    print_ai_output(ai_output)
    conversation.append({"role": "assistant", "content": ai_output})
    return conversation

# --- Run a short two-turn demo ---
mlflow.autolog()
with mlflow.start_run(run_name="GenAI_book_demo"):
    conversation = [{"role": "system", "content": "You are a helpful assistant."}]
    conversation = chat_once("Give me one fun fact about hummingbirds.", conversation)
    conversation = chat_once("Now make it shorter.", conversation)

2025/11/09 18:48:04 INFO mlflow.tracking.fluent: Autologging successfully enabled for openai.


You: Give me one fun fact about hummingbirds.
AI Assistant: One fun fact about hummingbirds is that they are the only birds that can fly backwards.
You: Now make it shorter.
AI Assistant: Hummingbirds are the only birds that can fly backwards.
